# 02 - Silver Merge and EDA

**Layer:** Silver (merged and cleaned data)

## Goal
Merge the 20 raw files into one master dataset, add `city` and `day_type`
extracted from the file names, and drop the leftover index column.

## Step 1: Setup
Ask for the bronze and silver folder paths and create the silver folder if needed.

In [1]:
import os
import pandas as pd

default_bronze = "../data/bronze/csv"
default_silver = "../data/silver"

bronze_path = input(f"Enter bronze folder path [{default_bronze}]: ").strip()
if bronze_path == "":
    bronze_path = default_bronze

silver_path = input(f"Enter silver folder path [{default_silver}]: ").strip()
if silver_path == "":
    silver_path = default_silver

# Create the silver folder if it does not exist
os.makedirs(silver_path, exist_ok=True)

# Collect and sort the CSV files
all_files = os.listdir(bronze_path)
csv_files = []
i = 0
while i < len(all_files):
    if all_files[i].lower().endswith(".csv"):
        csv_files.append(all_files[i])
    i += 1
csv_files.sort()
print(f"Found {len(csv_files)} CSV files")

Found 20 CSV files


## Step 2: Load, Tag, and Append
For each file: read it, extract `city` and `day_type` from the file name
(pattern: `{city}_{weekdays|weekends}.csv`), drop `Unnamed: 0`, and collect it.

In [2]:
frames = []
expected_rows = 0

i = 0
while i < len(csv_files):
    file_name = csv_files[i]
    df = pd.read_csv(os.path.join(bronze_path, file_name))
    expected_rows += len(df)

    # Extract city and day type from the file name
    name_parts = file_name[:-4].split("_")
    city = name_parts[0].capitalize()
    day_type = "weekday" if name_parts[1] == "weekdays" else "weekend"

    df["city"] = city
    df["day_type"] = day_type

    # Drop the leftover index column if it exists
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    frames.append(df)
    i += 1

master = pd.concat(frames, ignore_index=True)
print("Master shape:", master.shape)

Master shape: (51707, 21)


## Step 3: Validation
Check that no rows were lost, and that every city/day_type combination is present.

In [3]:
# Row count must match the sum of all files
print("Expected rows:", expected_rows)
print("Actual rows  :", len(master))
print("Row count matches:", expected_rows == len(master))

# Rows per city and day type
display(pd.crosstab(master["city"], master["day_type"]))

# Duplicates and nulls on the merged data
print("Duplicated rows:", master.duplicated().sum())
print("Total nulls:", master.isnull().sum().sum())

Expected rows: 51707
Actual rows  : 51707
Row count matches: True


day_type,weekday,weekend
city,,
Amsterdam,1103,977
Athens,2653,2627
Barcelona,1555,1278
Berlin,1284,1200
Budapest,2074,1948
Lisbon,2857,2906
London,4614,5379
Paris,3130,3558
Rome,4492,4535


Duplicated rows: 0
Total nulls: 0


## Step 4: Save the Merged Dataset
Save the merged data to the silver layer.

In [4]:
output_file = os.path.join(silver_path, "airbnb_merged.csv")
master.to_csv(output_file, index=False)
print("Saved:", output_file)

Saved: ../data/silver\airbnb_merged.csv
